# FinDocRAG — Week 1: Exploring FinanceBench

This notebook loads the **FinanceBench** dataset and explores it, so you understand the raw material before building anything.

It is written to run in **Google Colab** or locally. Just run the cells top to bottom.

Goal for this notebook: by the end, you should be able to describe what columns FinanceBench has and what a single question/answer example looks like.

## 1. Install dependencies
Only needed in Colab / a fresh environment.

In [1]:
!pip install datasets pandas pyarrow -q

## 2. Load FinanceBench
FinanceBench is a benchmark of expert-written questions and answers over real SEC filings. If the dataset id below is wrong, you will get a clear error — search https://huggingface.co/datasets for 'financebench' and update the id.

In [2]:
from datasets import load_dataset

DATASET_ID = "PatronusAI/financebench"   # change if needed

ds = load_dataset(DATASET_ID, split="train")
df = ds.to_pandas()
print(f"Loaded {len(df)} rows.")

/Users/majid/Desktop/RAG project/findoc-rag/notebooks/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 150/150 [00:00<00:00, 4856.39 examples/s]

Loaded 150 rows.


## 3. What columns does it have?
Always inspect the schema before writing code against a dataset.

In [3]:
print("Number of questions:", len(df))
print("Number of columns:", len(df.columns))
print()
for col in df.columns:
    print(" -", col)

Number of questions: 150
Number of columns: 15

 - financebench_id
 - company
 - doc_name
 - question_type
 - question_reasoning
 - domain_question_num
 - question
 - answer
 - justification
 - dataset_subset_label
 - evidence
 - gics_sector
 - doc_type
 - doc_period
 - doc_link


## 4. Look at a few full examples
This prints the first three rows, every column, so you can see exactly what one FinanceBench item contains.

In [4]:
for i in range(3):
    print("=" * 70)
    print(f"EXAMPLE {i}")
    print("=" * 70)
    for col in df.columns:
        value = str(df.iloc[i][col])
        if len(value) > 400:
            value = value[:400] + " ..."
        print(f"\n[{col}]")
        print(value)
    print()

EXAMPLE 0

[financebench_id]
financebench_id_03029

[company]
3M

[doc_name]
3M_2018_10K

[question_type]
metrics-generated

[question_reasoning]
Information extraction

[domain_question_num]
nan

[question]
What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

[answer]
$1577.00

[justification]
The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).

[dataset_subset_label]
OPEN_SOURCE

[evidence]
[{'evidence_text': 'Table of Contents \n3M Company and Subsidiaries\nConsolidated Statement of Cash Flow s\nYears ended December 31\n \n(Millions)\n \n2018\n \n2017\n \n2016\n \nCash Flows from Operating Activities\n \n \n \n \n \n \n \nNet income including noncontrolling interest\n \n$\n5,363 \n$\n4,869 \n$\n5,058 \nAdjustments to reconcile net income including noncon

## 5. How is the data distributed?
Understand the spread across companies and document types. Column names vary — this cell adapts to whatever columns exist.

In [6]:
# Show value counts for low-cardinality columns, skipping any that
# hold lists/arrays (which can't be counted this way).
for col in df.columns:
    try:
        nunique = df[col].nunique()
    except TypeError:
        print(f"--- {col}  (skipped: holds lists/arrays, not simple values) ---\n")
        continue
    if 1 < nunique <= 40:
        print(f"--- {col}  ({nunique} unique values) ---")
        print(df[col].value_counts())
        print()

--- company  (32 unique values) ---
company
PepsiCo                 11
Amcor                    9
Johnson & Johnson        9
3M                       8
AMD                      8
Best Buy                 8
Boeing                   8
American Express         7
MGM Resorts              7
Pfizer                   6
Ulta Beauty              6
Adobe                    5
JPMorgan                 5
Verizon                  5
Corning                  4
CVS Health               4
General Mills            4
Nike                     4
AES Corporation          3
Amazon                   3
American Water Works     3
Block                    3
Coca-Cola                3
Lockheed Martin          3
Walmart                  3
Activision Blizzard      2
Foot Locker              2
Microsoft                2
Netflix                  2
Costco                   1
Kraft Heinz              1
Paypal                   1
Name: count, dtype: int64

--- question_type  (3 unique values) ---
question_type
metrics-ge

## 6. What to take away

Answer these for yourself before moving on:

1. Which column holds the **question**? The **gold answer**?
2. Is there a column with the **evidence passage** (the text the answer comes from)? This is critical — it is what you will evaluate retrieval against in Phase 3.
3. Which column identifies the **source filing / company**?
4. How many distinct filings are there? (This decides how big your Week 2 slice is.)

**Next step (ROADMAP.md, Week 2):** pick ~10–15 filings and the questions tied to them, collect the filing text into `data/raw/`, and write a short `data/DATA_CARD.md`.

In [7]:
print("Distinct filings:", df["doc_name"].nunique())

Distinct filings: 84


In [8]:
# Keep only the big yearly reports (10-K)
tenk = df[df["doc_type"] == "10k"]

print("10-K rows (questions):", len(tenk))
print("Distinct 10-K filings:", tenk["doc_name"].nunique())
print()
print("Questions per filing (top 15):")
print(tenk["doc_name"].value_counts().head(15))

10-K rows (questions): 112
Distinct 10-K filings: 64

Questions per filing (top 15):
doc_name
AMD_2022_10K                   7
AMERICANEXPRESS_2022_10K       7
BOEING_2022_10K                7
PEPSICO_2022_10K               5
AMCOR_2023_10K                 4
3M_2022_10K                    3
AES_2022_10K                   3
BESTBUY_2023_10K               3
CVSHEALTH_2022_10K             3
JOHNSON_JOHNSON_2022_10K       3
PFIZER_2021_10K                3
VERIZON_2022_10K               3
3M_2018_10K                    2
ACTIVISIONBLIZZARD_2019_10K    2
ADOBE_2022_10K                 2
Name: count, dtype: int64
